# BiLSTM + attention (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_bilstm` (word indices + sequence lengths).

**Quick iteration:** Next cell sets `QUICK_ITERATION = True` for a fast smoke test. Set **`False`** for real training.

**Metrics:** Same proposal bundle as other starter notebooks.

In [5]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [6]:
import time

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_bilstm

QUICK_ITERATION = True

if QUICK_ITERATION:
    _train_n, _val_n = 2048, 512
    MAX_LEN = 64
    BATCH_SIZE = 128
    EPOCHS = 1
    MAX_VOCAB = 3000
    MIN_FREQ = 1
    HIDDEN = 64
else:
    _train_n, _val_n = None, None
    MAX_LEN = 256
    BATCH_SIZE = 64
    EPOCHS = 1
    MAX_VOCAB = 50_000
    MIN_FREQ = 2
    HIDDEN = 128

LR = 1e-3

data = preprocess_for_bilstm(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)
vocab_size = len(data.vocab)
embed_dim = 100
num_labels = len(LABEL_COLUMNS)
print("QUICK_ITERATION:", QUICK_ITERATION, "| Train/val:", data.X_train.shape, data.X_val.shape)

Device: mps
QUICK_ITERATION: True | Train/val: (2048, 64) (512, 64)


In [7]:
class AttentionBiLSTM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden: int,
        num_labels: int,
        padding_idx: int = 0,
    ):
        super().__init__()
        self.padding_idx = padding_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden * 2, 1)
        self.fc = nn.Linear(hidden * 2, num_labels)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x)
        lens = lengths.clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        scores = self.attn(out).squeeze(-1)
        positions = torch.arange(out.size(1), device=x.device).unsqueeze(0).expand_as(scores)
        mask = positions < lengths.unsqueeze(1).to(x.device)
        scores = scores.masked_fill(~mask, -1e9)
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx = (out * w).sum(dim=1)
        return self.fc(ctx)


model = AttentionBiLSTM(vocab_size, embed_dim, HIDDEN, num_labels).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.length_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

t0 = time.perf_counter()
model.train()
for epoch in range(EPOCHS):
    for xb, lenb, yb in train_loader:
        xb, lenb, yb = xb.to(DEVICE), lenb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb, lenb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
train_seconds = time.perf_counter() - t0
print(f"Training wall time ({EPOCHS} epoch(s)): {train_seconds:.1f} s")

Training wall time (1 epoch(s)): 11.2 s


In [8]:
def predict_logits_lengths(model, X_np, len_np, batch_size=512):
    model.eval()
    all_logits = []
    x = torch.tensor(X_np, dtype=torch.long)
    lens = torch.tensor(len_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE), lens[i : i + batch_size].to(DEVICE))
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)


logits_val = predict_logits_lengths(model, data.X_val, data.length_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
pred_val = (prob_val >= 0.5).astype(np.int64)

per_label_df, summary = multilabel_evaluation_report(data.y_val, pred_val, prob_val, LABEL_COLUMNS)
print("Micro / macro F1:", summary)
display(per_label_df)
for name, m in per_label_confusion_matrices(data.y_val, pred_val, LABEL_COLUMNS).items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary["f1_macro"] == 0.0 and summary["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common after 1 smoke epoch)."
        " ROC-AUC may still be > 0.5. Raise EPOCHS or set QUICK_ITERATION=False for real training."
    )

Micro / macro F1: {'f1_micro': 0.0, 'f1_macro': 0.0}


,label,precision,recall,f1,roc_auc
0,toxic,0.0,0.0,0.0,0.420370
1,severe_toxic,0.0,0.0,0.0,0.351378
2,obscene,0.0,0.0,0.0,0.410806
3,threat,0.0,0.0,0.0,0.594912
4,insult,0.0,0.0,0.0,0.422574
5,identity_hate,0.0,0.0,0.0,0.068047


toxic 
 [[457   0]
 [ 55   0]]
severe_toxic 
 [[508   0]
 [  4   0]]
obscene 
 [[485   0]
 [ 27   0]]
threat 
 [[511   0]
 [  1   0]]
insult 
 [[483   0]
 [ 29   0]]
identity_hate 
 [[507   0]
 [  5   0]]

--- Proposal summary (record for report / comparison) ---
  device: mps
  training_time_s: 11.25
  num_parameters: 385895
  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common after 1 smoke epoch). ROC-AUC may still be > 0.5. Raise EPOCHS or set QUICK_ITERATION=False for real training.


## Notes

- Increase `EPOCHS` and add validation-based **early stopping** for the final report.
- Tune **class weights** in `BCEWithLogitsLoss` (`pos_weight`) for rare heads (`threat`, `identity_hate`).